In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
with open("Data/language_acronyms_dict.pickle", "rb") as f:
    language_acronyms_dict = pickle.load(f)

df_data_2020 = pd.read_pickle("Data/df_data_2020_part_1.pickle")

In [4]:
text_cols = ["short_descr", "object_contract/title", "object descr titles", "ac cost criteria", "ac criteria"]

In [ ]:
#let's check to see if all values are present. This makes sense so lets move on
for col in df_data_2020.columns:
    if df_data_2020[col].value_counts().sum() < len(df_data_2020):
        print(col, ": ", len(df_data_2020) - df_data_2020[col].value_counts().sum())

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
from scipy.stats import anderson
import statsmodels.api as sm
import matplotlib.pyplot as plt
from ordered_set import OrderedSet
from sklearn.preprocessing import RobustScaler
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import copy
import re
import string
from langdetect import detect
from stop_words import safe_get_stop_words
from stop_words import get_stop_words
from nltk.tokenize import word_tokenize
import spacy
from transformers import XLMRobertaTokenizer
import torch

---------------------------------------------------------------------------------------------------------------------------------------
PRODUCING DATA AND MODELS FOR NOTEBOOK
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [7]:
stop_words_dict = {}

for language_acronym in language_acronyms_dict.keys():
    functionality = language_acronyms_dict[language_acronym][1]
    language = language_acronyms_dict[language_acronym][0]
    
    if functionality == "supported":
        stop_words_dict[language_acronym] = get_stop_words(language.lower())
    else:
        continue

In [13]:
language_models = {"ca": spacy.load("ca_core_news_sm"),
                   "el": spacy.load("el_core_news_sm"),
                   "en": spacy.load("en_core_web_sm"),
                   "es": spacy.load("es_core_news_sm"),
                   "it": spacy.load("it_core_news_sm"),
                   "nl": spacy.load("nl_core_news_sm"),
                   "pl": spacy.load("pl_core_news_sm"),
                   "sv": spacy.load("sv_core_news_sm")}

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING OF UNSTRUCTURED DATA
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [14]:
#we need to add a column stating the language, the country is unfortunately not sufficient to determine the language
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

In [15]:
def remove_stopwords_lemmatize(text: str, language: str, language_models=language_models, stop_words_dict=stop_words_dict, language_acronyms_dict=language_acronyms_dict):
    if language in stop_words_dict.keys():
        if language in language_models.keys(): #yes lemma
            #print("{} is in language models".format(language))
            if language_acronyms_dict[language][1] == "supported": #yes lemma, yes stopword
                nlp = language_models[language]
                stop_words = stop_words_dict.get(language, set())
                doc = nlp(text)
                clean_text_list = [token.lemma_ for token in doc if token.text.lower() not in stop_words]
                clean_text = ' '.join(clean_text_list)
                return clean_text
            else:                                                 #yes lemma, no stopword
                nlp = language_models[language]
                doc = nlp(text)
                clean_text_list = [token.lemma_ for token in doc]
                clean_text = ' '.join(clean_text_list)
                return clean_text
        else: #no lemma
            #print("{} is not in language models".format(language))
            if language_acronyms_dict[language][1] == "supported": #no lemma, yes stopwords
                stop_words = stop_words_dict.get(language, set())
                text_list = text.split()
                clean_text_list = [word for word in text_list if word not in stop_words]
                clean_text = ' '.join(clean_text_list)
                return clean_text
            else:                                                 #no lemma, no stopword
                return text
    else:                                                         #unknown language
        #print("{} is not known".format(language))
        return text

In [ ]:
test_cases = [["Dit is een voorbeeld voor no lemma, yes stopwords. stopwoorden zijn de het een ", "nl"],
              ["This is an example for yes lemma, yes stopwords. lemma: walk walking walked. stopwords: he she it the", "en"],
              ["Dit is een voorbeeld voor no lemma, no stopwords. stopwoorden zijn de het een", "BS"],
              ]

for test in test_cases:
    print(remove_stopwords_lemmatize(test[0], test[1]))

In [16]:
#let's preprocess the text columns by lowercasing, removing numbers and special characters, removing stop words and lemmatization
def clean_text(text, language):
    #to lowercase
    text = text.lower()
    #remove links
    pattern = r'https?://\S+|www\.\S+'
    text = re.sub(pattern, "", text)
    #remove punctuation
    pattern = r'[{}]'.format(re.escape(string.punctuation))
    text = re.sub(pattern, "", text)
    #remove next line
    pattern = r'[^ \w\.]'
    text = re.sub(pattern, "", text) 
    #remove words containing numbers
    pattern = r'\w*\d\w*'
    text = re.sub(pattern, '', text)
    text = remove_stopwords_lemmatize(text, language)
    return text

In [ ]:
test_cases = [["Dit is een voorbeeld voor no lemma, yes stopwords. stopwoorden zijn de het een ", "nl"],
              ["This is an example for yes lemma, yes stopwords. lemma: walk walking walked. stopwords: he she it the", "en"],
              ["Dit is een voorbeeld voor no lemma, no stopwords. stopwoorden zijn de het een", "BS"],
              ]

for test in test_cases:
    print(clean_text(test[0], test[1]))

In [31]:
df["language"] = df[text_cols[0]].apply(detect_language)

In [10]:
#with open("Data/df_languages_column.pickle", "wb") as f:
#    pickle.dump(df, f)

In [14]:
df = pd.read_pickle("Data/df_languages_column.pickle")

In [17]:
cols = ['original index',
         'language',
         'country',
         'cpv_code',
         'type_contract',
         'procedure',
         'joint_procurement_involved',
         'central_purchasing',
         'eu_fund',
         'framework or dynamic purchasing',
         'contracting authority type',
         'contracting authority activity',
         'nb_tenders_received',
         'nb_tenders_received_sme',
         'award_contract/val_total: 0',
         'duration',
         'ac_price_weighting: 0',
         'short_descr',
         'ac criteria',
         'object_contract/title',
         'object descr titles',
         'ac cost criteria',
         ]
df = df[cols]

IDEA FOR LATER (THOUGHT OF AT 01-11), IS TO USE THE COUNTRY AS LANGUAGE IN CASE OF UNKNOWN.

In [ ]:
#Let's inspect how many of the rows have no detected language. 0.042 have a language that is not recognized, not too bad
percentage_unknown_language = len(df.loc[df["language"] == "unknown"]) / len(df) * 100
print("{}% of row have a language that is not recognized".format(percentage_unknown_language), df["language"].value_counts())

In [43]:
for col in text_cols:
    df[col] = df[col].astype(str)

In [ ]:
for col in text_cols:
    for i in tqdm(range(0, len(df))):
        text = df[col].iloc[i]
        language = df["language"].iloc[i]
        df.at[i, col] = clean_text(text, language)

In [ ]:
df["short_descr"].loc[df["language"] == "nl"].iloc[4]

In [152]:
#with open("Data/df_nlp_processed", "wb") as f:
#    pickle.dump(df, f)

language detection: final result of the peprocessing, of the 229352 rows, all have a language, but 21521 are unknown ~10%.<br>
stopwords: 188921 rows are in a language which is supported by the stopwords dict <br>
lemmatization: 76194 rows have been lemmatized by the language models. <br>

We can always go back and upload more models and stop words dicts

In [ ]:
df["language"].value_counts()["nl"]

In [ ]:
stop_words_row_count = 0 

for language in df["language"].value_counts().keys():
    if language in stop_words_dict.keys():
            stop_words_row_count += df["language"].value_counts()[language]
    else:
        continue

stop_words_row_count

In [ ]:
lemmatization_row_count = 0

for language in df["language"].value_counts().keys():
    if language in language_models.keys():
            lemmatization_row_count += df["language"].value_counts()[language]
    else:
        continue
    
lemmatization_row_count

---------------------------------------------------------------------------------------------------------------------------------------
VECTORIZATION OF UNSTRUCTURED DATA
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer

In [53]:
with open("Data/df_nlp_processed", "rb") as f:
    df = pickle.load(f)

In [6]:
text_cols = ["short_descr", "object_contract/title", "object descr titles", "ac cost criteria", "ac criteria"]

In [55]:
#let's check the length of all text columns
list_of_texts = {"short_descr": [], 
                 "object_contract/title": [], 
                 "object descr titles": [], 
                 "ac cost criteria": [], 
                 "ac criteria": []}
for col in text_cols:
    cell_of_text = []
    for i in range(0, len(df)):
        list_of_texts[col].append(df[col].iloc[i].split())

In [56]:
#1929 texts are too long, but how much too long? Most of them really are way too long, we could look at sentence embedding seperate parts and then averaging, but for now
#let's move on and take the trimming for granted.
text_too_long = []

for col in list_of_texts.keys():
    for tokenized_text in list_of_texts[col]:
        if len(tokenized_text) > 256:
            text_too_long.append(tokenized_text)

len(text_too_long)

past_limit_amount = []
for tokenized_text in text_too_long:
    past_limit_amount.append(len(tokenized_text) - 256)

In [ ]:
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

In [ ]:
sentences = ['This framework generates embeddings for each input sentence',
    'Sentences are passed as a list of string.',
    'The quick brown fox jumps over the lazy dog.']

embeddings = list(model.encode(sentences))
len(embeddings[0])

In [ ]:
col_texts_dict = {}

for col in text_cols:
    col_texts_dict[col] = df_test[col].values.tolist()

embeddings_dict = {"short_descr": [], 
                 "object_contract/title": [], 
                 "object descr titles": [], 
                 "ac cost criteria": [], 
                 "ac criteria": []}

for col in embeddings_dict.keys():
    sentences = col_texts_dict[col]
    embeddings_col = list(model.encode(sentences))
    embeddings_dict[col].append(embeddings_col)

In [7]:
with open("Data/embeddings.pickle", "rb") as f:
    embeddings = pickle.load(f)

In [37]:
for col in embeddings.keys():
    df[col] = [embedding for embedding in embeddings[col][0]]

In [57]:
#last thing to preprocess is the date_conclusion_contract
date_col = []
for i in range(0, len(df)):
    if pd.isna(df["date_conclusion_contract"].iloc[i]) == False and df["date_conclusion_contract"].iloc[i] != None:
        date_string = df["date_conclusion_contract"].iloc[i].split(".")[0]
        date_col.append(date_string)
    else:
        date_col.append(np.nan)

df["date_conclusion_contract"] = date_col

---------------------------------------------------------------------------------------------------------------------------------------
SCALING OF CONTINUOUS COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

DO NOT FORGET TO TRAIN/TEST/SPLIT BEFORE SCALING, AND DO NOT SCALE THE TARGET VARIABLE

In [58]:
#let's check if the continuous columns are normally distributed
continuous_cols = ["nb_tenders_received", "nb_tenders_received_sme", "award_contract/val_total: 0"]
data = df["award_contract/val_total: 0"].values.tolist()

In [59]:
df['award_contract/val_total: 0'] = df['award_contract/val_total: 0'].astype(float)

Using a shapiro-wilks test allows us to find statistical significance easier with larger sample size (unadvised above N>5000) so we will not use it.

In [ ]:
for col in continuous_cols:
    result = anderson(df[col])
    if result.statistic < result.critical_values[2]:
        print("Data appears to be normally distributed for column: {}".format(col))
    else:
        print("Data does not appear to be normally distributed for column: {}".format(col))

    sm.qqplot(df[col], line='s')
    plt.title("Q-Q Plot for {}".format(col))
    plt.show()

In [61]:
continuous_cols_arrays = [[value for value in df[continuous_cols[0]].values.tolist() if np.isnan(value) != True], 
                          [value for value in df[continuous_cols[1]].values.tolist() if np.isnan(value) != True], 
                          [value for value in df[continuous_cols[2]].values.tolist() if np.isnan(value) != True]]

In [ ]:
for i, col in enumerate(continuous_cols):
    value_array = continuous_cols_arrays[i]
    boxplot, ax1 = plt.subplots()
    ax1.set_title("Distribution of {}".format(col))
    ax1.boxplot(value_array)
    #plt.show()
    plt.savefig("Figures/Distribution of {}.png".format(str(col).replace("/", "-")));

The data is very clearly not normally distributed for all three continuous variables and the boxplot show that all three variables have some insane outliers, right long tail. Let's reduce the outliers. Reducing the upper bound to 95% is already a huge difference, much more suitable for the number of tenders boxplots. The val_total boxplot still has a lot of outliers

In [63]:
continuous_cols

['nb_tenders_received',
 'nb_tenders_received_sme',
 'award_contract/val_total: 0']

In [ ]:
upper_bound = 0.95
lower_bound = 0.00
upper_bound_dict = {}

for i, array in enumerate(continuous_cols_arrays):    
    lower_bound_indice = int(lower_bound * len(array))
    upper_bound_indice = int(upper_bound * len(array))
    value_array = np.sort(array)[lower_bound_indice:upper_bound_indice]
    upper_bound_dict[continuous_cols[i]] = np.sort(array)[upper_bound_indice: upper_bound_indice + 1]

    boxplot, ax1 = plt.subplots()
    ax1.set_title("Distribution of {}".format(continuous_cols[i]))
    ax1.boxplot(value_array)
    #plt.show() 

In [ ]:
upper_bound_dict

In [ ]:
#let's remove the the rows that belong to the top 5% for all continuous columns. 20775 rows when cutting the top 5% that is quite steep but necessary. We can probably get away with less cutting for tenders received
rows_for_delete = []
for col in upper_bound_dict.keys():
    for i, row_index in enumerate(range(0, len(df))):
        if df[col].iloc[i] > upper_bound_dict[col]:
            rows_for_delete.append(i)
            
rows_for_delete = list(OrderedSet(rows_for_delete))
df = df.drop(labels = rows_for_delete, axis = 0).reset_index(drop=True)
print(len(rows_for_delete))

In [ ]:
#let's check how many of the offers have values and impute if possible. 30290 values or 13.3% are missing. Let's check the procedure for these cases as that is likely to influence it
amount_tenders_no_offers = len(df.loc[pd.isna(df["nb_tenders_received"]) == True])
amount_tenders_no_offers / len(df)

In [ ]:
#let's first filter the framework or dynamic purchasing as this is known to have no bids
yes_framework = [index for index in df.index.tolist() if pd.isna(df["nb_tenders_received"].iloc[index]) == True and
                  df["framework or dynamic purchasing"].iloc[index] != "no framework or dynamic purchasing"]

#now let's get all those rows where the procedure is PT_AWARD_CONTRACT_WITHOUT_CALL but has no framework or dynamic purchasing
award_no_call = [index for index in df.index.tolist() if pd.isna(df["nb_tenders_received"].iloc[index]) == True and
                 df["procedure"].iloc[index] == "PT_AWARD_CONTRACT_WITHOUT_CALL" and 
                 df["framework or dynamic purchasing"].iloc[index] == "no framework or dynamic purchasing"]

print("of the {} cases with no tender, {} can be explained by having no call for tender/purchasing agreement/dps".format(amount_tenders_no_offers, 
    len(yes_framework) + len(award_no_call)))

In [ ]:
#this leaves too many cases unaccounted for so let's check other procedures but "open", but only grants three cases so it has nothing to do with the procedure
df["framework or dynamic purchasing"].value_counts()

In [70]:
#award_procedure_other = [index for index in df.index.tolist() if pd.isna(df["nb_tenders_received"].iloc[index]) == True and
#                 df["procedure"].iloc[index] != "PT_AWARD_CONTRACT_WITHOUT_CALL" and 
#                 df["procedure"].iloc[index] != "PT_OPEN" and
#                 df["framework or dynamic purchasing"].iloc[index] == "no framework or dynamic purchasing"]

#len(award_procedure_other)

In [ ]:
#are there rows where the number of offers for sme is present but not for other offers, answer is no
len([index for index in df.index.tolist() if pd.isna(df["nb_tenders_received"].iloc[index]) == True and
                  pd.isna(df["nb_tenders_received_sme"].iloc[index] != True)])

THE TWO CODE CELLS BELOW USE MEAN IMPUTATION BUT THAT IS PROBABLY NOT SMART, ALSO NOT BECAUSE WE STILL NEED TO IMPUTE THE DURATION, IF THAT IS SOMETHING WE WANT TO DO ANYWAY

In [ ]:
#let's look at how many unique cpv_codes we have. that is quite a lot so let's go for divisions only.
cpv_codes_divisions = set([code[:2] for code in df["cpv_code"].values])
cpv_codes_groups = set([code[:3] for code in df["cpv_code"].values])
cpv_codes_classes = set([code[:4] for code in df["cpv_code"].values])

print("divisions: ", len(cpv_codes_divisions), "\n", "groups: ", len(cpv_codes_groups), "\n", "classes: ", len(cpv_codes_classes))

In [73]:
for i in range(0, len(df)):
    value = df["cpv_code"].iloc[i]
    if pd.isna(value) != True:
        df.at[i, "cpv_code"] = str(value)[:4]

In [ ]:
len(df["cpv_code"].unique())

In [75]:
df["date_conclusion_contract"] = pd.to_datetime(df["date_conclusion_contract"], format = "%Y-%m-%d")

In [76]:
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme"]
categorical_columns = base_n_encoder_cols + one_hot_columns

In [ ]:
pd.isna(df["contracting authority type"]).any()

In [ ]:
df.head()

In [79]:
with open("Data/df_before_scalling.pickle", "wb") as f:
    pickle.dump(df, f)